In [40]:
import smartsheet, datetime

smartsheet_client = smartsheet.Smartsheet('w2wMKac07W77sYoUf9TaIyMH2UMeVGjPI4YL3')



In [41]:
def get_cell_by_column_id(row, column_id):
    # Retrieves a cell from a row by column ID.
    return row.get_column(column_id)

In [42]:
# API Endpoint
URL = "https://app.smartsheet.com/sheets/gv9mXfv6q3GWW3gHF4mmfv5QccXv5W5qGqhrVmP1"
SHEET_ID = 2720818803986308

In [43]:
# The API identifies columns by Id, but it's more convenient to refer to column names. Store a map here
column_map = {}

def initialize_column_map(sheet):    
    for column in sheet.columns:
        if column.id is None:
            print(f"Warning: Column '{column.title}' not found in the sheet. Skipping...")
            continue
        column_map[column.title] = column.id
        print(f"Column ID for '{column.title}': {column.id}")

initialize_column_map(smartsheet_client.Sheets.get_sheet(SHEET_ID))

Column ID for 'Primary Column': 6323173360095108
Column ID for 'Column2': 4071373546409860
Column ID for 'Column3': 8574973173780356
Column ID for 'Column4': 412198849171332
Column ID for 'Column5': 5535836329955204
Column ID for 'Column6': 1032236702584708
Column ID for 'Column7': 8350586097061764
Column ID for 'Column8': 3846986469691268
Column ID for 'Column9': 6098786283376516
Column ID for 'Column10': 1595186656006020
Column ID for 'Column11': 7224686190219140
Column ID for 'Column12': 2721086562848644
Column ID for 'Column13': 4972886376533892
Column ID for 'Column14': 469286749163396
Column ID for 'Column15': 8632061073772420
Column ID for 'Column16': 4128461446401924
Column ID for 'Column17': 6380261260087172
Column ID for 'Column18': 1876661632716676
Column ID for 'Column19': 7506161166929796


In [44]:
get_cell_by_column_id(smartsheet_client.Sheets.get_sheet(SHEET_ID).rows[0], column_map["Column3"])

In [45]:
print(get_cell_by_column_id(smartsheet_client.Sheets.get_sheet(SHEET_ID).rows[0], column_map["Column3"]))

{"columnId": 8574973173780356, "displayValue": "Waiting on Shipping", "value": "Waiting on Shipping"}


In [46]:
# In Smartsheet, each cell object typically has both a 'value' and a 'displayValue' attribute.
# - 'value': The raw value stored in the cell (e.g., a number, date, or text as entered).
# - 'displayValue': The formatted value as shown in the Smartsheet UI (e.g., with currency symbols, date formatting, or formulas resolved).

# Let's fetch a cell and print both attributes to illustrate the difference:
row = smartsheet_client.Sheets.get_sheet(SHEET_ID).rows[0]
cell = get_cell_by_column_id(row, column_map["Column3"])
status_value = getattr(cell, 'display_value', None)

print("Raw value:", getattr(cell, 'value', None))
print("Display value:", status_value)

Raw value: Waiting on Shipping
Display value: Waiting on Shipping


In [47]:
if status_value is None:
    print("Status value is None (cell may be empty or missing).")
else:
    print(f"Status value is present: {status_value}")

Status value is present: Waiting on Shipping


In [48]:
new_cell = smartsheet_client.models.Cell()
new_cell.column_id = column_map["Column11"]
new_cell.value = 0

print(new_cell)

{"columnId": 7224686190219140, "value": 0}


In [49]:
updates_cell = get_cell_by_column_id(row, new_cell.column_id)
updates_value = getattr(updates_cell, 'value', None)

# Prepare the new update string
update_note = f"{datetime.datetime.now().strftime('%m/%d/%Y')}: Accessed via API -MC"

In [50]:
# If the cell is empty, start a new list; if it's a list, append; if it's a string, convert to list
if updates_value is None:
    new_updates_value = [update_note]
elif isinstance(updates_value, list):
    new_updates_value = updates_value + [update_note]
else:
    new_updates_value = [str(updates_value), update_note]

In [51]:
# Combine the existing updates_value and the new update_note into a single string (with line breaks)
if updates_value is None:
    combined_updates = update_note
else:
    combined_updates = f"{updates_value}\n{update_note}"

# Create a new Cell object for the update
cell_to_update = smartsheet_client.models.Cell()
cell_to_update.column_id = new_cell.column_id  # Column11
cell_to_update.value = combined_updates

# Create a Row object with the updated cell
row_to_update = smartsheet_client.models.Row()
row_to_update.id = row.id
row_to_update.cells = [cell_to_update]

# Update the row in Smartsheet
response = smartsheet_client.Sheets.update_rows(SHEET_ID, [row_to_update])
print("Update response:", response)

Update response: {"data": [{"cells": [{"columnId": 6323173360095108, "displayValue": "07/03/25 10:22 AM", "value": "07/03/25 10:22 AM"}, {"columnId": 4071373546409860, "displayValue": "Sound Transit", "value": "Sound Transit"}, {"columnId": 8574973173780356, "displayValue": "Waiting on Shipping", "value": "Waiting on Shipping"}, {"columnId": 412198849171332, "displayValue": "NCR - 00122", "value": "NCR - 00122"}, {"columnId": 5535836329955204, "displayValue": "NA", "value": "NA"}, {"columnId": 1032236702584708, "displayValue": "IDPS", "value": "IDPS"}, {"columnId": 8350586097061764, "displayValue": "S4IDPS24002", "value": "S4IDPS24002"}, {"columnId": 3846986469691268, "displayValue": "R1C011322", "value": "R1C011322"}, {"columnId": 6098786283376516, "displayValue": "SIL4 said to not use this serial number because of updating issues.", "value": "SIL4 said to not use this serial number because of updating issues."}, {"columnId": 1595186656006020, "displayValue": "SIL4", "value": "SIL4"},